# MPMGame workflow on an ANDES-derived LFT model map

This notebook folds the original `mpm-andes-demo` workflow into `masked-perturbation-model`. The pipeline is:

1. acquire/load an ANDES benchmark case,
2. extract or construct a stable state-space model,
3. expose read/write channels,
4. derive the transfer-map \(M:w\mapsto r\) through the reusable LFT construction `mpmgame.lft.build_model_map`,
5. construct static attack candidates from frequency-domain scans of \(M\),
6. enumerate admissible defense masks,
7. compute success sets, dominated actions, the reduced game, and mixed strategies.

For static attack success/failure checks, this notebook keeps the same numerically reliable test used in the standalone demo:

\[
A_{\mathrm{cl}}(\Delta,\nabla)=A+B_w(\Delta\circ\nabla)C_r.
\]

The LFT-derived transfer matrix \(M(s)=C_r(sI-A)^{-1}B_w+D_{rw}\) is the object used to construct attack candidates and vulnerability scans; the equivalent static state-space closure is used to test destabilization robustly.

## 0. Setup

From the repository root, install the package and optional ANDES/Jupyter dependencies:

```bash
pip install -e ".[andes,demos]"
```

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if SRC_ROOT.exists() and str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

PROJECT_ROOT

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import control as ct

import mpmgame as mpm

from mpm_grid_demo.config import DemoConfig
from mpm_grid_demo.acquire import clone_andes_cases, find_case_files
from mpm_grid_demo.load_case import load_andes_system, setup_power_flow_and_dynamics
from mpm_grid_demo.linearize import (
    extract_andes_state_matrix,
    synthetic_two_area_like_model,
    verify_stability,
)
from mpm_grid_demo.channels import (
    select_speed_reads_and_power_writes,
    select_random_channels,
)
from mpm_grid_demo.bridge import build_lft_from_linear_model
from mpm_grid_demo.game_workflow import (
    frequency_grid,
    freqresp_matrix,
    approximate_hinf_norm,
    single_link_vulnerability_heatmap,
    top_k_single_link_attacks,
    rank1_static_delta_from_hinf,
    single_link_static_delta,
    closed_loop_eigs_static,
    is_destabilizing_static,
    compute_success_sets_static,
    payoff_matrix_static,
    eliminate_dominated_strategies_static,
    defense_dataframe,
    payoff_dataframe,
)
from mpm_grid_demo.plots import plot_eigs, plot_heatmap, plot_game_matrix

print("Using mpmgame from:", mpm.__file__)
config = DemoConfig(project_root=PROJECT_ROOT)
config

## 1. Acquire ANDES cases

This clones the public `andes_cases` repository into `data/raw/andes_cases` if it is not already present. The default keyword is `ieee39`, so the notebook starts from an IEEE-style ANDES case when available.

In [ ]:
try:
    case_repo = clone_andes_cases(config)
    print(f"Case repo: {case_repo}")
except Exception as exc:
    case_repo = config.case_repo_dir
    print("Could not clone cases automatically.")
    print("Reason:", repr(exc))
    print("You can manually clone:")
    print(f"git clone {config.case_repo_url} {case_repo}")

case_files = find_case_files(config.case_repo_dir, keyword=config.preferred_case_keyword)
print(f"Found {len(case_files)} candidate files")
for path in case_files[:20]:
    print(path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path)

## 2. Load benchmark and extract/construct a linear model

The goal is to obtain a stable small-signal model. If ANDES extraction fails, the notebook falls back to the same synthetic two-area-like model used by the standalone demo so the complete MPM/game workflow remains runnable.

In [ ]:
linear_model = None
system = None

if case_files:
    case_path = case_files[0]
    print(f"Trying case file: {case_path}")
    try:
        system = load_andes_system(case_path, setup=False)
        system = setup_power_flow_and_dynamics(system)
        linear_model = extract_andes_state_matrix(system)
        print("Successfully extracted ANDES small-signal model.")
    except Exception as exc:
        print("ANDES extraction did not complete.")
        print("Reason:", repr(exc))

if linear_model is None and config.use_synthetic_fallback:
    print("Using synthetic stable two-area-like fallback model.")
    linear_model = synthetic_two_area_like_model()

if linear_model is None:
    raise RuntimeError("No linear model was constructed. Enable synthetic fallback or patch ANDES extraction.")

A = np.asarray(linear_model.A, dtype=float)
state_names = linear_model.state_names

print(f"Model source: {linear_model.source}")
print(f"A shape: {A.shape}")
print("First states:", state_names[:10])

In [ ]:
stable, eigvals = verify_stability(A, tol=config.stability_tol)
print("Nominal strictly stable:", stable)
print("Max Re(lambda):", float(np.max(np.real(eigvals))))
plot_eigs(eigvals, title=f"Nominal eigenvalues: {linear_model.source}")
plt.show()

## 3. Select read/write channels

The standalone demo used four random read/write channels for the full finite game. That same default is preserved here. A physically motivated alternative based on speed reads and power writes is left one line away for power-system-specific experiments.

In [ ]:
channels = select_random_channels(A, state_names, n_channels=4, random_seed=config.random_seed)
# channels = select_speed_reads_and_power_writes(A, state_names)

B_w = np.asarray(channels.B_w, dtype=float)
C_r = np.asarray(channels.C_r, dtype=float)
D_rw = np.asarray(channels.D_rw, dtype=float)
read_metadata = channels.read_metadata
write_metadata = channels.write_metadata

print("B_w shape:", B_w.shape)
print("C_r shape:", C_r.shape)
print("D_rw shape:", D_rw.shape)

print("\nReads:")
for read in read_metadata:
    print("  ", read)

print("\nWrites:")
for write in write_metadata:
    print("  ", write)

## 4. Build \(M:w\mapsto r\) through the LFT construction

`build_lft_from_linear_model(...)` now delegates to `mpmgame.lft.build_model_map(...)`. Its return value is the transfer-function matrix \(M\), represented as a `numpy.ndarray` of SISO `control.TransferFunction` entries.

In [ ]:
M = build_lft_from_linear_model(linear_model, channels)

print("M shape (reads × writes):", M.shape)
print("number of reads:", M.shape[0])
print("number of writes:", M.shape[1])
print("number of states in source realization:", A.shape[0])
print("M[0, 0]:")
print(M[0, 0])

In [ ]:
omega_probe = 1.0
M_jw = freqresp_matrix(M, omega_probe)
print("M(jω) shape:", M_jw.shape)
print("max |entry| at ω=1:", float(np.max(np.abs(M_jw))))

## 5. Frequency-domain vulnerability scan

This section constructs the same finite attack candidate set as the standalone demo: one rank-one static attack derived from the approximate peak singular vectors of \(M\), and several single-link static attacks drawn from the largest gridded entrywise vulnerabilities.

In [ ]:
omega_grid = frequency_grid(config.frequency_min, config.frequency_max, config.frequency_points)
hinf_result = approximate_hinf_norm(M, omega_grid)

print("Approx ||M||_inf:", hinf_result.norm)
print("Critical frequency:", hinf_result.critical_frequency)
print("Critical attack size ~ 1/||M||_inf:", 1.0 / hinf_result.norm)

In [ ]:
V = single_link_vulnerability_heatmap(M, omega_grid)
V_df = pd.DataFrame(
    V,
    index=[write["name"] for write in write_metadata],
    columns=[read["name"] for read in read_metadata],
)
V_df

In [ ]:
plot_heatmap(
    V,
    title="Single-link vulnerability heatmap",
    xlabel="Read channel",
    ylabel="Write channel",
)
plt.show()

## 6. Construct static attack candidates

In [ ]:
attacks = []

rank1_delta = rank1_static_delta_from_hinf(hinf_result, safety_factor=config.attack_gain_safety_factor)
attacks.append(mpm.AttackAction(label=f"rank1_static_sf{config.attack_gain_safety_factor:.2f}", delta=rank1_delta))

top_links = top_k_single_link_attacks(V, config.top_k_single_link_attacks)
for write_idx, read_idx, vulnerability in top_links:
    delta = single_link_static_delta(
        write_index=write_idx,
        read_index=read_idx,
        vulnerability_value=vulnerability,
        safety_factor=1.10,
        n_writes=M.shape[1],
        n_reads=M.shape[0],
    )
    attacks.append(
        mpm.AttackAction(
            label=f"single_w{write_idx}_r{read_idx}_sf1.10",
            delta=delta,
        )
    )

print(f"Constructed {len(attacks)} static attacks.")
for attack in attacks:
    destabilizes_nominal = is_destabilizing_static(A, B_w, C_r, attack.delta, pole_tol=config.stability_tol)
    print(attack.label, "destabilizes unmasked system:", destabilizes_nominal)

## 7. Enumerate admissible defense masks

In [ ]:
c_w = np.ones(M.shape[1])
c_r = np.ones(M.shape[0])

defense_masks = mpm.admissible_defenses(
    num_writes=M.shape[1],
    num_reads=M.shape[0],
    c_w=c_w,
    c_r=c_r,
    budget=config.defense_budget,
    include_empty=False,
)

defenses = [
    mpm.DefenseAction(label=f"D{k:03d}", mask=mask)
    for k, mask in enumerate(defense_masks)
]
defense_df = defense_dataframe(defenses, c_w=c_w, c_r=c_r)

print(f"Generated {len(defenses)} admissible defense masks.")
defense_df.head(20)

## 8. Compute success sets

In [ ]:
success = compute_success_sets_static(A, B_w, C_r, attacks, defenses, pole_tol=config.stability_tol)

print("Attack success sets:")
for label, defense_labels in success.attack_success.items():
    print(label, sorted(defense_labels))

print("\nDefense success sets:")
for label, attack_labels in success.defense_success.items():
    print(label, sorted(attack_labels))

## 9. Identify dominated actions and construct the reduced game

In [ ]:
dominated_attack_labels = mpm.dominated_attacks(success.attack_success)
dominated_defense_labels = mpm.dominated_defenses(success.defense_success)

print("Dominated attacks:", sorted(dominated_attack_labels))
print("Dominated defenses:", sorted(dominated_defense_labels))

reduced = eliminate_dominated_strategies_static(A, B_w, C_r, attacks, defenses, pole_tol=config.stability_tol)
print("Reduced attacks:", [attack.label for attack in reduced.attacks])
print("Reduced defenses:", [defense.label for defense in reduced.defenses])
print("Reduced payoff shape:", reduced.payoff.shape)

reduced_payoff_df = payoff_dataframe(reduced.payoff, reduced.attacks, reduced.defenses)
reduced_payoff_df

## 10. Full payoff matrix and visualization

In [ ]:
U = payoff_matrix_static(A, B_w, C_r, attacks, defenses, pole_tol=config.stability_tol)
U_df = payoff_dataframe(U, attacks, defenses)
U_df

In [ ]:
plot_game_matrix(U_df)
plt.show()

## 11. Solve the reduced zero-sum game

In [ ]:
attacker_mix, defender_mix, game_value = mpm.solve_zero_sum_game(reduced.payoff)

attacker_strategy = pd.Series(attacker_mix, index=[attack.label for attack in reduced.attacks], name="attacker_probability")
defender_strategy = pd.Series(defender_mix, index=[defense.label for defense in reduced.defenses], name="defender_probability")

print("Game value:", game_value)
print("\nAttacker mixed strategy:")
display(attacker_strategy)
print("\nDefender mixed strategy:")
display(defender_strategy)

## 12. Closed-loop eigenvalue check for a representative attack

This final diagnostic mirrors the standalone demo's emphasis that success/failure is defined by static closed-loop instability for the channelized state-space realization.

In [ ]:
representative_attack = attacks[0]
representative_eigs = closed_loop_eigs_static(A, B_w, C_r, representative_attack.delta)
print("Representative attack:", representative_attack.label)
print("Max Re(lambda) under unmasked static closure:", float(np.max(np.real(representative_eigs))))
plot_eigs(representative_eigs, title=f"Closed-loop eigenvalues: {representative_attack.label}")
plt.show()